# 03 · Evaluate (base vs fine-tuned)

Score both models on the held-out **test** split with real tool-calling metrics, then compare.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Niikhi/peft-function-calling.git"
REPO_DIR = "peft-function-calling"
if not os.path.exists("src") and not os.path.exists(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(os.path.join(REPO_DIR, "src")):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())


In [ ]:
!pip -q install unsloth matplotlib seaborn

In [ ]:
from src.config import load_config
from src.data import load_jsonl
from src.evaluate import make_hf_generate_fn, run_eval
from src.model import load_model_and_tokenizer, for_inference
from huggingface_hub import snapshot_download
import gc, torch, os

cfg = load_config()

adapter_dir = str(cfg.path("adapter_dir"))
if not os.path.exists(os.path.join(adapter_dir, "config.json")):
    snapshot_download(repo_id="Nikrobber/qwen2.5-3b-tool-calling", local_dir=adapter_dir)

test_recs = load_jsonl(cfg.file_in("data_dir", "test.jsonl"))

def eval_one(weights, label):
    cfg.model["base_id"] = weights
    model, tok = load_model_and_tokenizer(cfg); for_inference(model)
    m = run_eval(cfg, test_recs, tok, make_hf_generate_fn(cfg, model, tok), label)
    del model; gc.collect(); torch.cuda.empty_cache()
    return m

base_id = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
results = {
    "base":      eval_one(base_id, "base"),
    "finetuned": eval_one(adapter_dir, "finetuned"),
}


### Comparison charts

In [ ]:
from src.visualize import plot_metric_comparison, plot_error_breakdown
from IPython.display import Image
plot_error_breakdown(results, cfg.file_in("figures_dir", "error_breakdown.png"))
Image(plot_metric_comparison(results, cfg.file_in("figures_dir", "metric_comparison.png")))

In [ ]:
import pandas as pd
pd.DataFrame(results).T[["exact_match","name_accuracy","argument_f1","hallucination_rate","count_accuracy"]]